# IEEE-CIS Fraud Detection — Random Forest

In [ ]:
!pip install -q dagshub mlflow
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import mlflow
import mlflow.sklearn
import dagshub
from kaggle_secrets import UserSecretsClient
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

# MLflow setup
user_secrets = UserSecretsClient()
MLFLOW_TRACKING_PASSWORD = user_secrets.get_secret('ML_token1')
MLFLOW_TRACKING_USERNAME = user_secrets.get_secret('username')
repo_name = 'IEEE-Fraud-Detection'

mlflow.set_tracking_uri(f'https://dagshub.com/{MLFLOW_TRACKING_USERNAME}/{repo_name}.mlflow')
os.environ['MLFLOW_TRACKING_USERNAME'] = MLFLOW_TRACKING_USERNAME
os.environ['MLFLOW_TRACKING_PASSWORD'] = MLFLOW_TRACKING_PASSWORD
dagshub.init(repo_name=repo_name, repo_owner=MLFLOW_TRACKING_USERNAME, mlflow=True)
mlflow.set_experiment('RandomForest_Training')
print('✅ MLflow connected - Experiment: RandomForest_Training')

## Cleaning

In [ ]:
# Load data
train_trans = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_id    = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')
test_trans  = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv')
test_id     = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv')

# Merge transaction + identity
train = train_trans.merge(train_id, on='TransactionID', how='left')
test  = test_trans.merge(test_id,   on='TransactionID', how='left')

# Separate target
y         = train['isFraud']
train_ids = train['TransactionID']
test_ids  = test['TransactionID']

X_train = train.drop(['TransactionID', 'isFraud'], axis=1)
X_test  = test.drop(['TransactionID'], axis=1)

print(f'Train shape: {X_train.shape}')
print(f'Test shape:  {X_test.shape}')
print(f'Fraud rate:  {y.mean():.2%}')

In [ ]:
# Drop columns with >90% missing
missing_pct = X_train.isnull().sum() / len(X_train)
drop_cols   = missing_pct[missing_pct > 0.9].index.tolist()
print(f'Dropping {len(drop_cols)} columns with >90% missing')
X_train = X_train.drop(drop_cols, axis=1, errors='ignore')
X_test  = X_test.drop(drop_cols,  axis=1, errors='ignore')

# Drop high-cardinality categorical columns (>100 unique values)
cat_cols  = X_train.select_dtypes(include=['object']).columns
high_card = [col for col in cat_cols if X_train[col].nunique() > 100]
print(f'Dropping {len(high_card)} high-cardinality columns')
X_train = X_train.drop(high_card, axis=1, errors='ignore')
X_test  = X_test.drop(high_card,  axis=1, errors='ignore')

# Align columns
X_test = X_test.reindex(columns=X_train.columns)

print(f'Final train shape: {X_train.shape}')
print(f'Final test shape:  {X_test.shape}')

In [ ]:
# Reduce memory usage
def reduce_mem_usage(df):
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object:
            c_min, c_max = df[col].min(), df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min  and c_max < np.iinfo(np.int8).max:  df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max: df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max: df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max: df[col] = df[col].astype(np.float32)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max: df[col] = df[col].astype(np.float32)
    return df

X_train = reduce_mem_usage(X_train)
X_test  = reduce_mem_usage(X_test)
print('✅ Memory reduced')

In [ ]:
# Log cleaning info
with mlflow.start_run(run_name='RandomForest_Cleaning'):
    mlflow.log_params({
        'train_samples':        len(X_train),
        'test_samples':         len(X_test),
        'fraud_rate':           round(float(y.mean()), 4),
        'n_features_after_drop': X_train.shape[1],
        'dropped_missing_cols': len(drop_cols),
        'dropped_highcard_cols': len(high_card),
    })
    mlflow.log_metric('n_features', X_train.shape[1])
print('✅ Cleaning logged to MLflow')

## Feature Engineering

In [ ]:
# Label-encode remaining categorical columns
cat_cols_remaining = X_train.select_dtypes(include=['object']).columns.tolist()
le_dict = {}
for col in cat_cols_remaining:
    le = LabelEncoder()
    combined = pd.concat([X_train[col], X_test[col]], axis=0).astype(str)
    le.fit(combined)
    X_train[col] = le.transform(X_train[col].astype(str))
    X_test[col]  = le.transform(X_test[col].astype(str))
    le_dict[col] = le
print(f'Label-encoded {len(cat_cols_remaining)} columns')

In [ ]:
# Transaction hour and day from TransactionDT
# TransactionDT is seconds offset from a reference time
START_DATE = pd.Timestamp('2017-12-01')
for df in [X_train, X_test]:
    df['Transaction_hour'] = (df['TransactionDT'] / 3600) % 24
    df['Transaction_day']  = (df['TransactionDT'] / (3600 * 24)) % 7
    df['Transaction_week'] = (df['TransactionDT'] / (3600 * 24 * 7)).astype(int)

# Transaction amount features
for df in [X_train, X_test]:
    df['TransactionAmt_log']   = np.log1p(df['TransactionAmt'])
    df['TransactionAmt_cents'] = df['TransactionAmt'] - df['TransactionAmt'].astype(int)

print('✅ Feature engineering done')
print(f'Shape after FE: {X_train.shape}')

In [ ]:
# Log feature engineering info
with mlflow.start_run(run_name='RandomForest_FeatureEngineering'):
    mlflow.log_params({
        'label_encoded_cols':   len(cat_cols_remaining),
        'added_time_features':  3,
        'added_amount_features': 2,
        'total_features':       X_train.shape[1],
    })
    mlflow.log_metric('n_features_after_fe', X_train.shape[1])
print('✅ Feature Engineering logged to MLflow')

## Feature Selection

In [ ]:
# Fill NaN before feature selection (RF cannot handle NaN)
X_train_filled = X_train.fillna(-999)
X_test_filled  = X_test.fillna(-999)

# Quick RF to get feature importances
from sklearn.ensemble import RandomForestClassifier
selector_rf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
selector_rf.fit(X_train_filled, y)

importances = pd.Series(selector_rf.feature_importances_, index=X_train.columns)
importances = importances.sort_values(ascending=False)

# Plot top 30
plt.figure(figsize=(12, 8))
importances.head(30).plot(kind='bar')
plt.title('Top 30 Feature Importances (Random Forest)')
plt.tight_layout()
plt.savefig('/kaggle/working/rf_feature_importances.png')
plt.show()
print('✅ Feature importances computed')

In [ ]:
# Keep top N features by importance threshold
threshold      = 0.0001
selected_cols  = importances[importances >= threshold].index.tolist()
print(f'Features above threshold {threshold}: {len(selected_cols)}')

X_train_sel = X_train_filled[selected_cols]
X_test_sel  = X_test_filled[selected_cols]
print(f'Shape after selection: {X_train_sel.shape}')

In [ ]:
# Log feature selection to MLflow
with mlflow.start_run(run_name='RandomForest_FeatureSelection'):
    mlflow.log_params({
        'selection_method':     'RandomForest_importance',
        'importance_threshold': threshold,
        'n_features_selected':  len(selected_cols),
        'n_features_dropped':   X_train_filled.shape[1] - len(selected_cols),
    })
    mlflow.log_metric('n_selected_features', len(selected_cols))
    mlflow.log_artifact('/kaggle/working/rf_feature_importances.png')
print('✅ Feature Selection logged to MLflow')

## Training

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
# ── v1: Baseline (shallow trees, few estimators) ─────────────────────────────
# Intentionally underfitted — shows what happens with too-simple config
params_v1 = {
    'n_estimators': 100,
    'max_depth':    5,
    'min_samples_leaf': 50,
    'random_state': 42,
    'n_jobs':       -1,
    'class_weight': 'balanced',
}
model_v1   = RandomForestClassifier(**params_v1)
cv_scores  = []
for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_sel, y)):
    X_tr, X_val = X_train_sel.iloc[train_idx], X_train_sel.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    model_v1.fit(X_tr, y_tr)
    preds = model_v1.predict_proba(X_val)[:, 1]
    auc   = roc_auc_score(y_val, preds)
    cv_scores.append(auc)
    print(f'Fold {fold+1} AUC: {auc:.5f}')

mean_auc_v1 = np.mean(cv_scores)
std_auc_v1  = np.std(cv_scores)
print(f'\n✅ v1 Mean CV AUC: {mean_auc_v1:.5f} (+/- {std_auc_v1:.5f})')
print('   → Expected: underfitted due to shallow depth + high min_samples_leaf')

with mlflow.start_run(run_name='RandomForest_v1_underfitted'):
    mlflow.log_params(params_v1)
    mlflow.log_params({'note': 'intentionally_underfitted'})
    mlflow.log_metrics({'cv_auc_mean': mean_auc_v1, 'cv_auc_std': std_auc_v1})
    mlflow.sklearn.log_model(model_v1, 'model')

In [ ]:
# ── v2: Overfitted (very deep trees, no regularisation) ───────────────────────
params_v2 = {
    'n_estimators':     200,
    'max_depth':        None,   # unlimited depth → likely overfit
    'min_samples_leaf': 1,
    'random_state':     42,
    'n_jobs':           -1,
}
model_v2    = RandomForestClassifier(**params_v2)
cv_scores_v2 = []
train_aucs_v2 = []
for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_sel, y)):
    X_tr, X_val = X_train_sel.iloc[train_idx], X_train_sel.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    model_v2.fit(X_tr, y_tr)
    train_preds = model_v2.predict_proba(X_tr)[:, 1]
    val_preds   = model_v2.predict_proba(X_val)[:, 1]
    train_auc   = roc_auc_score(y_tr, train_preds)
    val_auc     = roc_auc_score(y_val, val_preds)
    train_aucs_v2.append(train_auc)
    cv_scores_v2.append(val_auc)
    print(f'Fold {fold+1}  Train AUC: {train_auc:.5f}  Val AUC: {val_auc:.5f}')

mean_auc_v2  = np.mean(cv_scores_v2)
std_auc_v2   = np.std(cv_scores_v2)
mean_train_v2 = np.mean(train_aucs_v2)
print(f'\n✅ v2 Mean CV AUC: {mean_auc_v2:.5f} (+/- {std_auc_v2:.5f})')
print(f'   Mean Train AUC: {mean_train_v2:.5f}')
print(f'   → Gap = {mean_train_v2 - mean_auc_v2:.5f}  (large gap = overfitting)')

with mlflow.start_run(run_name='RandomForest_v2_overfitted'):
    mlflow.log_params(params_v2)
    mlflow.log_params({'note': 'intentionally_overfitted_no_max_depth'})
    mlflow.log_metrics({
        'cv_auc_mean':   mean_auc_v2,
        'cv_auc_std':    std_auc_v2,
        'train_auc_mean': mean_train_v2,
        'overfit_gap':   round(mean_train_v2 - mean_auc_v2, 5),
    })
    mlflow.sklearn.log_model(model_v2, 'model')

In [ ]:
# ── v3: Balanced / well-tuned ─────────────────────────────────────────────────
params_v3 = {
    'n_estimators':     300,
    'max_depth':        15,
    'min_samples_leaf': 10,
    'max_features':     'sqrt',
    'random_state':     42,
    'n_jobs':           -1,
    'class_weight':     'balanced',
}
model_v3    = RandomForestClassifier(**params_v3)
cv_scores_v3 = []
train_aucs_v3 = []
for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_sel, y)):
    X_tr, X_val = X_train_sel.iloc[train_idx], X_train_sel.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    model_v3.fit(X_tr, y_tr)
    train_preds = model_v3.predict_proba(X_tr)[:, 1]
    val_preds   = model_v3.predict_proba(X_val)[:, 1]
    train_aucs_v3.append(roc_auc_score(y_tr, train_preds))
    cv_scores_v3.append(roc_auc_score(y_val, val_preds))
    print(f'Fold {fold+1}  Train AUC: {train_aucs_v3[-1]:.5f}  Val AUC: {cv_scores_v3[-1]:.5f}')

mean_auc_v3  = np.mean(cv_scores_v3)
std_auc_v3   = np.std(cv_scores_v3)
mean_train_v3 = np.mean(train_aucs_v3)
print(f'\n✅ v3 Mean CV AUC: {mean_auc_v3:.5f} (+/- {std_auc_v3:.5f})')
print(f'   Mean Train AUC: {mean_train_v3:.5f}')
print(f'   → Gap = {mean_train_v3 - mean_auc_v3:.5f}')

with mlflow.start_run(run_name='RandomForest_v3_balanced'):
    mlflow.log_params(params_v3)
    mlflow.log_params({'note': 'balanced_depth_regularisation'})
    mlflow.log_metrics({
        'cv_auc_mean':    mean_auc_v3,
        'cv_auc_std':     std_auc_v3,
        'train_auc_mean': mean_train_v3,
        'overfit_gap':    round(mean_train_v3 - mean_auc_v3, 5),
    })
    mlflow.sklearn.log_model(model_v3, 'model')

In [ ]:
# ── v4: More estimators + stricter regularisation ─────────────────────────────
params_v4 = {
    'n_estimators':       500,
    'max_depth':          12,
    'min_samples_leaf':   20,
    'min_samples_split':  10,
    'max_features':       0.5,
    'max_samples':        0.8,
    'random_state':       42,
    'n_jobs':             -1,
    'class_weight':       'balanced_subsample',
}
model_v4    = RandomForestClassifier(**params_v4)
cv_scores_v4 = []
train_aucs_v4 = []
for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_sel, y)):
    X_tr, X_val = X_train_sel.iloc[train_idx], X_train_sel.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    model_v4.fit(X_tr, y_tr)
    train_preds = model_v4.predict_proba(X_tr)[:, 1]
    val_preds   = model_v4.predict_proba(X_val)[:, 1]
    train_aucs_v4.append(roc_auc_score(y_tr, train_preds))
    cv_scores_v4.append(roc_auc_score(y_val, val_preds))
    print(f'Fold {fold+1}  Train AUC: {train_aucs_v4[-1]:.5f}  Val AUC: {cv_scores_v4[-1]:.5f}')

mean_auc_v4  = np.mean(cv_scores_v4)
std_auc_v4   = np.std(cv_scores_v4)
mean_train_v4 = np.mean(train_aucs_v4)
print(f'\n✅ v4 Mean CV AUC: {mean_auc_v4:.5f} (+/- {std_auc_v4:.5f})')
print(f'   Mean Train AUC: {mean_train_v4:.5f}')
print(f'   → Gap = {mean_train_v4 - mean_auc_v4:.5f}')

with mlflow.start_run(run_name='RandomForest_v4_strict_reg'):
    mlflow.log_params(params_v4)
    mlflow.log_params({'note': 'strict_regularisation_max_samples'})
    mlflow.log_metrics({
        'cv_auc_mean':    mean_auc_v4,
        'cv_auc_std':     std_auc_v4,
        'train_auc_mean': mean_train_v4,
        'overfit_gap':    round(mean_train_v4 - mean_auc_v4, 5),
    })
    mlflow.sklearn.log_model(model_v4, 'model')

In [ ]:
# ── Compare all versions ──────────────────────────────────────────────────────
results = pd.DataFrame({
    'Version':  ['v1_underfitted', 'v2_overfitted', 'v3_balanced', 'v4_strict_reg'],
    'Val_AUC':  [mean_auc_v1, mean_auc_v2, mean_auc_v3, mean_auc_v4],
    'Std':      [std_auc_v1,  std_auc_v2,  std_auc_v3,  std_auc_v4],
})
print(results.to_string(index=False))

plt.figure(figsize=(8, 4))
plt.bar(results['Version'], results['Val_AUC'], yerr=results['Std'], capsize=5)
plt.ylim(0.8, 1.0)
plt.title('Random Forest — CV AUC by Version')
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('/kaggle/working/rf_version_comparison.png')
plt.show()

best_auc  = max(mean_auc_v1, mean_auc_v2, mean_auc_v3, mean_auc_v4)
best_idx  = np.argmax([mean_auc_v1, mean_auc_v2, mean_auc_v3, mean_auc_v4])
best_params = [params_v1, params_v2, params_v3, params_v4][best_idx]
best_name   = ['v1_underfitted', 'v2_overfitted', 'v3_balanced', 'v4_strict_reg'][best_idx]
print(f'\n🏆 Best version: {best_name}  AUC: {best_auc:.5f}')

In [ ]:
# ── Train final model on full data & register ─────────────────────────────────
final_model = RandomForestClassifier(**best_params)
final_model.fit(X_train_sel, y)

# Save preprocessing info
preprocessing_info = {
    'drop_cols':      drop_cols,
    'high_card':      high_card,
    'le_dict':        le_dict,
    'selected_cols':  selected_cols,
    'feature_cols':   X_train_sel.columns.tolist(),
}
with open('/kaggle/working/rf_preprocessing.pkl', 'wb') as f:
    pickle.dump(preprocessing_info, f)

with mlflow.start_run(run_name='RandomForest_Final_Model') as run:
    mlflow.log_params(best_params)
    mlflow.log_params({'best_version': best_name})
    mlflow.log_metrics({
        'final_cv_auc_mean': best_auc,
    })
    mlflow.log_artifact('/kaggle/working/rf_preprocessing.pkl')
    mlflow.log_artifact('/kaggle/working/rf_version_comparison.png')
    mlflow.sklearn.log_model(
        final_model,
        artifact_path='rf_final_pipeline',
        registered_model_name='RandomForest_FraudDetection',
    )
    run_id = run.info.run_id
    print(f'✅ Run ID: {run_id}')

print(f'✅ Best version: {best_name}')
print(f'✅ Best CV AUC:  {best_auc:.5f}')
print("✅ Model registered as 'RandomForest_FraudDetection'")